# Video Daily Stats Pipeline

- Daily data
- Continuous extraction
- Raw JSON → Volume
- Volume → Bronze Delta

In [0]:
# %pip install google-api-python-client python-dotenv
# %restart_python

## 1. Create Volume and incoming folder

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS youtube_content_intelligence.bronze.vol_video_daily_stats;

In [0]:
dbutils.fs.mkdirs("/Volumes/youtube_content_intelligence/bronze/vol_video_daily_stats/incoming/")

## 2. Extract video daily stats

In [0]:
from src.extraction.video_daily_stats import extract_video_daily_stats

video_stats = extract_video_daily_stats()

## 3. Write raw JSON to Volume

In [0]:
import json

collection_date = video_stats[0]["collection_date"]

volume_path = f"/Volumes/youtube_content_intelligence/bronze/vol_video_daily_stats/incoming/video_daily_stats_{collection_date}.json"

with open(volume_path, "w") as file:
    json.dump(video_stats, file, indent=2)

## 4. Verify raw JSON

In [0]:
display(dbutils.fs.ls("/Volumes/youtube_content_intelligence/bronze/vol_video_daily_stats/incoming/"))

In [0]:
display(dbutils.fs.head(volume_path, 1000))

## 5. Load raw JSON with Auto Loader

In [0]:
from pyspark.sql.functions import current_timestamp

df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", "/Volumes/youtube_content_intelligence/bronze/vol_video_daily_stats/schema/")
    .load("/Volumes/youtube_content_intelligence/bronze/vol_video_daily_stats/incoming/")
    .withColumn("ingested_at", current_timestamp())
)

In [0]:
df.printSchema()

## 6. Write to Bronze Delta

In [0]:
query = (
    df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/youtube_content_intelligence/bronze/vol_video_daily_stats/checkpoint/")
    .trigger(availableNow=True)
    .toTable("youtube_content_intelligence.bronze.brz_video_daily_stats")
)

## 7. Validate Bronze table

In [0]:
%sql
SELECT *
FROM youtube_content_intelligence.bronze.brz_video_daily_stats;

In [0]:
%sql
DESCRIBE TABLE youtube_content_intelligence.bronze.brz_video_daily_stats;

In [0]:
%sql
SELECT COUNT(*) AS row_count
FROM youtube_content_intelligence.bronze.brz_video_daily_stats;